# Lesson 2 - Exercise 2: Analyzing GQA Memory Savings by Comparing Llama-3.2-1B and GPT-2 XL

**Goal:** Understand and quantify the memory efficiency of GQA by comparing the KV cache sizes of Llama-3.2-1B (GQA) and GPT-2 XL (MHA).

## 1. Imports and Configuration

In [1]:
import os
import torch
from transformers import AutoConfig, AutoModelForCausalLM # AutoModel for dtype inference

# --- Configuration ---
# Model 1: Expected to use GQA (local workspace copy if present, else the ungated Hub mirror)
_local_llama = "/voc/shared/models/llama/Llama-3.2-1B"
model_name_gqa = os.environ.get("UDACI_MODEL", _local_llama if os.path.isdir(_local_llama) else "unsloth/Llama-3.2-1B")
# Model 2: Expected to use MHA
model_name_mha = "openai-community/gpt2-xl"

# Loading the full 1.5B-parameter GPT-2 XL (6 GB fp32) just to read its dtype is wasteful;
# set LOAD_FULL_MODELS=1 to do it anyway. By default we read the dtype from the config.
LOAD_FULL_MODELS = os.environ.get("LOAD_FULL_MODELS", "0") == "1"

print(f"Comparing GQA Model: {model_name_gqa}")
print(f"With MHA Model: {model_name_mha}")


Comparing GQA Model: unsloth/Llama-3.2-1B
With MHA Model: openai-community/gpt2-xl


## 3. Analysis Helper Function

You will complete the `analyze_model_kv_cache` function below. This function should:
1. Load the model's configuration.
2. (Recommended) Temporarily load the model itself to accurately infer its `dtype`, then delete the model to free resources.
3. Extract key parameters: `num_hidden_layers`, `num_attention_heads` (query heads), `num_key_value_heads`, `hidden_size`.
4. Determine the size in bytes of the model's data type.
5. Calculate the `head_dim`.
6. Identify the attention mechanism type (MHA, MQA, or GQA) and the GQA grouping factor if applicable.
7. Calculate the KV cache memory added **per layer, per generated token**.
8. Calculate the **total** KV cache memory added **per generated token (across all layers)**.
9. Return these values in a dictionary.

In [2]:
def analyze_model_kv_cache(model_name_to_analyze, model_label):
    print(f"\n--- Analyzing {model_label}: {model_name_to_analyze} ---")
    config = None
    model_dtype = None

    # TODO 1: Load model configuration and infer dtype
    try:
        if not LOAD_FULL_MODELS:
            raise RuntimeError("LOAD_FULL_MODELS=0 -> skipping full model load")
        model = AutoModelForCausalLM.from_pretrained(model_name_to_analyze, low_cpu_mem_usage=True)
        config = model.config
        model_dtype = next(model.parameters()).dtype
        print(f"Full model loaded. Actual dtype: {model_dtype}")
        del model
    except Exception as e:
        print(f"Could not load full model for {model_name_to_analyze} to infer dtype ({e}). Loading config only.")
        config = AutoConfig.from_pretrained(model_name_to_analyze)
        # transformers >= 4.5x exposes `dtype`; older versions `torch_dtype`. GPT-2's config carries neither,
        # in which case from_pretrained() would materialise fp32 weights by default.
        model_dtype = getattr(config, "dtype", None) or getattr(config, "torch_dtype", None) or torch.float32
        if isinstance(model_dtype, str):
            model_dtype = getattr(torch, model_dtype)
        print(f"Config loaded. Assumed/Config dtype: {model_dtype}")

    # TODO 2: Extract parameters from the configuration object (config)
    # GPT-2 uses n_layer / n_head / n_embd; Llama uses num_hidden_layers / num_attention_heads / hidden_size.
    num_hidden_layers = getattr(config, "num_hidden_layers", None) or config.n_layer
    num_attention_heads_query = getattr(config, "num_attention_heads", None) or config.n_head
    # MHA models do not define num_key_value_heads -> equal to the number of query heads
    num_key_value_heads_actual = getattr(config, "num_key_value_heads", None) or num_attention_heads_query
    hidden_size_model = getattr(config, "hidden_size", None) or config.n_embd

    # TODO 3: Determine dtype_size_bytes based on model_dtype (2 for float16/bfloat16, 4 for float32)
    dtype_size_bytes = torch.tensor([], dtype=model_dtype).element_size()

    print(f"\n  --- Extracted Configuration for {model_label} ---")

    # TODO 4: Calculate head_dim (hidden_size_model / num_attention_heads_query)
    head_dim = getattr(config, "head_dim", None) or hidden_size_model // num_attention_heads_query
    head_dim = int(head_dim)

    # TODO 5: Determine attention_type_str (MHA, MQA, or GQA) and gqa_group_factor if GQA
    gqa_group_factor = None
    if num_key_value_heads_actual == num_attention_heads_query:
        attention_type_str = "MHA (Multi-Head Attention)"
    elif num_key_value_heads_actual == 1:
        attention_type_str = "MQA (Multi-Query Attention)"
    else:
        attention_type_str = "GQA (Grouped-Query Attention)"
        gqa_group_factor = num_attention_heads_query // num_key_value_heads_actual

    print(f"  Identified Attention Type: {attention_type_str}"
          + (f" — {gqa_group_factor} query heads share each K/V head" if gqa_group_factor else ""))

    # TODO 6: Calculate actual KV cache size PER LAYER, PER TOKEN in bytes
    size_actual_per_layer_per_token_bytes = 2 * num_key_value_heads_actual * head_dim * dtype_size_bytes

    # TODO 7: Calculate TOTAL actual KV cache size PER TOKEN (across all layers)
    total_size_actual_per_token_bytes = size_actual_per_layer_per_token_bytes * num_hidden_layers

    results = {
        "model_name": model_name_to_analyze,
        "label": model_label,
        "L": num_hidden_layers,
        "N_q": num_attention_heads_query,
        "N_kv_actual": num_key_value_heads_actual,
        "D_head": head_dim,
        "dtype_size_bytes": dtype_size_bytes,
        "attention_type": attention_type_str,
        "cache_per_layer_per_token_bytes": size_actual_per_layer_per_token_bytes,
        "total_cache_per_token_bytes": total_size_actual_per_token_bytes,
        "gqa_group_factor": gqa_group_factor
    }

    print(f"  Number of Hidden Layers (L):           {results['L']}")
    print(f"  Number of Query Heads (N_q):         {results['N_q']}")
    print(f"  Number of Key/Value Heads (N_kv):    {results['N_kv_actual']}")
    print(f"  Model Hidden Size (D_model):         {hidden_size_model}")
    print(f"  Data Type Size (bytes):              {results['dtype_size_bytes']}")
    print(f"  Calculated Head Dimension (D_head):    {results['D_head']}")
    return results


## 4. Analyze Both Models

In [3]:
results_gqa_model = analyze_model_kv_cache(model_name_gqa, "Llama-3.2-1B (GQA)")
results_mha_model = analyze_model_kv_cache(model_name_mha, "GPT-2 XL (MHA)")


--- Analyzing Llama-3.2-1B (GQA): unsloth/Llama-3.2-1B ---
Could not load full model for unsloth/Llama-3.2-1B to infer dtype (LOAD_FULL_MODELS=0 -> skipping full model load). Loading config only.


Config loaded. Assumed/Config dtype: torch.bfloat16

  --- Extracted Configuration for Llama-3.2-1B (GQA) ---
  Identified Attention Type: GQA (Grouped-Query Attention) — 4 query heads share each K/V head
  Number of Hidden Layers (L):           16
  Number of Query Heads (N_q):         32
  Number of Key/Value Heads (N_kv):    8
  Model Hidden Size (D_model):         2048
  Data Type Size (bytes):              2
  Calculated Head Dimension (D_head):    64

--- Analyzing GPT-2 XL (MHA): openai-community/gpt2-xl ---
Could not load full model for openai-community/gpt2-xl to infer dtype (LOAD_FULL_MODELS=0 -> skipping full model load). Loading config only.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Config loaded. Assumed/Config dtype: torch.float32

  --- Extracted Configuration for GPT-2 XL (MHA) ---
  Identified Attention Type: MHA (Multi-Head Attention)
  Number of Hidden Layers (L):           48
  Number of Query Heads (N_q):         25
  Number of Key/Value Heads (N_kv):    25
  Model Hidden Size (D_model):         1600
  Data Type Size (bytes):              4
  Calculated Head Dimension (D_head):    64


## 5. Prediction Step

Before proceeding to the detailed comparison, answer these questions based on your initial understanding and the configurations you just extracted (or will extract once your `analyze_model_kv_cache` function is complete).

In [4]:
print("\n\n--- Prediction Step ---")
# 1. Llama-3.2-1B: N_q = 32, N_kv = 8  ->  GQA with grouping factor 32/8 = 4.
#    Its per-layer KV cache should be 4x smaller than a hypothetical MHA version of itself
#    (which would store 32 K/V heads instead of 8).
# 2. GPT-2 XL: N_q = 25, N_kv = 25 (no num_key_value_heads in the config) -> classic MHA.
# 3. Comparison: Llama stores 8 heads x 64 dims x 2 bytes (bf16) = 1 KB per K (and per V) per layer,
#    GPT-2 XL stores 25 heads x 64 dims x 4 bytes (fp32) = 6.25 KB. So Llama's per-layer-per-token
#    cache should be ~6x smaller (2x from dtype, ~3x from GQA head count); even at equal dtype it
#    is 25/8 ≈ 3.1x smaller. GPT-2 XL also has 3x more layers (48 vs 16), so the *total* per-token
#    cache gap should be even larger (~18x at native dtypes, ~9x at equal dtype).
r_g, r_m = results_gqa_model, results_mha_model
print(f"Llama-3.2-1B: N_q={r_g['N_q']}, N_kv={r_g['N_kv_actual']} -> predicted {r_g['attention_type']}, "
      f"group factor {r_g['gqa_group_factor']}, expected {r_g['N_q']/r_g['N_kv_actual']:.0f}x smaller per-layer cache than an MHA twin.")
print(f"GPT-2 XL:     N_q={r_m['N_q']}, N_kv={r_m['N_kv_actual']} -> predicted {r_m['attention_type']}.")
print("Prediction: Llama-3.2-1B's per-layer-per-token KV cache is SMALLER than GPT-2 XL's — fewer KV heads "
      "(8 vs 25) at the same head_dim (64) and a smaller dtype (bf16 vs fp32).")




--- Prediction Step ---
Llama-3.2-1B: N_q=32, N_kv=8 -> predicted GQA (Grouped-Query Attention), group factor 4, expected 4x smaller per-layer cache than an MHA twin.
GPT-2 XL:     N_q=25, N_kv=25 -> predicted MHA (Multi-Head Attention).
Prediction: Llama-3.2-1B's per-layer-per-token KV cache is SMALLER than GPT-2 XL's — fewer KV heads (8 vs 25) at the same head_dim (64) and a smaller dtype (bf16 vs fp32).


## 6. Detailed Analysis & Comparison

In [5]:
def print_analysis_results(res, label_override=None):
    if not res:
        print(f"Could not analyze results for {label_override or 'model'}")
        return

    label = label_override or res["label"]
    print(f"\n--- KV Cache Analysis for {label} ---")
    print(f"  Attention Type: {res['attention_type']}")
    print(f"  Parameters: L={res['L']}, N_q={res['N_q']}, N_kv_actual={res['N_kv_actual']}, D_head={res['D_head']}, dtype_size={res['dtype_size_bytes']} bytes")
    print(f"  Actual KV Cache Memory Per Layer, Per Token: {res['cache_per_layer_per_token_bytes']} Bytes "
          f"({res['cache_per_layer_per_token_bytes']/1024:.2f} KB)")
    print(f"  Total Actual KV Cache Memory Per Token (all layers): {res['total_cache_per_token_bytes']/(1024*1024):.4f} MB")

    # Internal saving factor for GQA/MQA models
    if res['N_kv_actual'] < res['N_q'] and res['N_kv_actual'] > 0:
        internal_saving = res['N_q'] / res['N_kv_actual']
        print(f"  Internal Saving Factor (vs. its own hypothetical MHA): {internal_saving:.2f}x")

print("\n\n--- DETAILED ANALYSIS & COMPARISON ---")
if results_gqa_model:
    print_analysis_results(results_gqa_model)
if results_mha_model:
    print_analysis_results(results_mha_model)

if results_gqa_model and results_mha_model:
    gqa_cache_plt = results_gqa_model['cache_per_layer_per_token_bytes']
    mha_cache_plt = results_mha_model['cache_per_layer_per_token_bytes']

    print("\n--- Direct Comparison (Per Layer, Per Token KV Cache) ---")
    print(f"  {results_gqa_model['label']}: {gqa_cache_plt} Bytes/layer/token ({gqa_cache_plt/1024:.2f} KB)")
    print(f"  {results_mha_model['label']}: {mha_cache_plt} Bytes/layer/token ({mha_cache_plt/1024:.2f} KB)")

    if gqa_cache_plt < mha_cache_plt:
        comparison_factor = mha_cache_plt / gqa_cache_plt
        print(f"  Observation: {results_gqa_model['label']}'s per-layer cache is ~{comparison_factor:.2f}x smaller than {results_mha_model['label']}'s.")
    else:
        comparison_factor = gqa_cache_plt / mha_cache_plt
        print(f"  Observation: {results_gqa_model['label']}'s per-layer cache is ~{comparison_factor:.2f}x LARGER than {results_mha_model['label']}'s.")
    # Decompose the factor into its ingredients
    head_factor = results_mha_model['N_kv_actual'] / results_gqa_model['N_kv_actual']
    dim_factor = results_mha_model['D_head'] / results_gqa_model['D_head']
    dtype_factor = results_mha_model['dtype_size_bytes'] / results_gqa_model['dtype_size_bytes']
    print(f"  Decomposition: KV-head count {head_factor:.2f}x  ×  head_dim {dim_factor:.2f}x  ×  dtype {dtype_factor:.2f}x "
          f"= {head_factor*dim_factor*dtype_factor:.2f}x")
    print(f"  At an equal dtype the architectural (GQA) advantage alone is {head_factor*dim_factor:.2f}x.")

    print("\n--- Practical Implication - Max Sequence Length Estimation ---")
    cache_gqa_total_mb_per_token = results_gqa_model['total_cache_per_token_bytes'] / (1024*1024)
    cache_mha_total_mb_per_token = results_mha_model['total_cache_per_token_bytes'] / (1024*1024)
    available_vram_for_cache_gb = 6 # Example: 6GB VRAM available JUST for KV cache
    available_vram_for_cache_mb = available_vram_for_cache_gb * 1024

    print(f"\nAssuming {available_vram_for_cache_gb} GB of VRAM is available *exclusively for the KV cache*:")
    if cache_gqa_total_mb_per_token > 0:
        max_tokens_gqa = available_vram_for_cache_mb / cache_gqa_total_mb_per_token
        print(f"  {results_gqa_model['label']} (L={results_gqa_model['L']}) could theoretically support ~{int(max_tokens_gqa)} tokens.")

    if cache_mha_total_mb_per_token > 0:
        max_tokens_mha = available_vram_for_cache_mb / cache_mha_total_mb_per_token
        print(f"  {results_mha_model['label']} (L={results_mha_model['L']}) could theoretically support ~{int(max_tokens_mha)} tokens.")

    print("\n--- Discussion ---")
    print(f"  Total per-token cache = per-layer cost × L. GPT-2 XL pays {results_mha_model['L']/results_gqa_model['L']:.0f}x more layers AND "
          f"{head_factor:.2f}x more K/V heads per layer (both at head_dim {results_gqa_model['D_head']}), so its per-token cache is "
          f"{cache_mha_total_mb_per_token/cache_gqa_total_mb_per_token:.1f}x larger and it fits {cache_mha_total_mb_per_token/cache_gqa_total_mb_per_token:.1f}x fewer tokens in the same VRAM.")
    print("  The number of K/V heads (GQA vs MHA) is the lever that scales the cache without touching model quality much,"
          " which is why modern models (Llama-3, Mistral, Qwen2) all use GQA; depth (L) is set by the desired model capacity, and dtype"
          " gives a further 2x (bf16) or 4x (int8 KV cache) on top. Together these decide how long a context — and how many concurrent"
          " users — a given GPU can actually serve.")

print("\nExercise Complete.")




--- DETAILED ANALYSIS & COMPARISON ---

--- KV Cache Analysis for Llama-3.2-1B (GQA) ---
  Attention Type: GQA (Grouped-Query Attention)
  Parameters: L=16, N_q=32, N_kv_actual=8, D_head=64, dtype_size=2 bytes
  Actual KV Cache Memory Per Layer, Per Token: 2048 Bytes (2.00 KB)
  Total Actual KV Cache Memory Per Token (all layers): 0.0312 MB
  Internal Saving Factor (vs. its own hypothetical MHA): 4.00x

--- KV Cache Analysis for GPT-2 XL (MHA) ---
  Attention Type: MHA (Multi-Head Attention)
  Parameters: L=48, N_q=25, N_kv_actual=25, D_head=64, dtype_size=4 bytes
  Actual KV Cache Memory Per Layer, Per Token: 12800 Bytes (12.50 KB)
  Total Actual KV Cache Memory Per Token (all layers): 0.5859 MB

--- Direct Comparison (Per Layer, Per Token KV Cache) ---
  Llama-3.2-1B (GQA): 2048 Bytes/layer/token (2.00 KB)
  GPT-2 XL (MHA): 12800 Bytes/layer/token (12.50 KB)
  Observation: Llama-3.2-1B (GQA)'s per-layer cache is ~6.25x smaller than GPT-2 XL (MHA)'s.
  Decomposition: KV-head count 3